# 06 — Evaluation
**Steps 23–26:** Test the model, generate confusion matrix, compute accuracy metrics, and report 2025 results.

In [ ]:
import os, sys
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from sklearn.metrics import (
    confusion_matrix, classification_report,
    accuracy_score, precision_score, recall_score, f1_score
)
from torch_geometric.nn import GATConv

DEVICE        = 'cuda' if torch.cuda.is_available() else 'cpu'
PROCESSED_DIR = os.path.join('..', 'data', 'processed')
OUTPUTS_DIR   = os.path.join('..', 'outputs')
CLASS_NAMES   = ['Water', 'Trees', 'Crops', 'Built-up Areas', 'Bare Ground']

In [ ]:
# ── Re-define & Load Model ────────────────────────────────────
class ModalityBranch(nn.Module):
    def __init__(self, in_dim, hidden_dim, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim), nn.BatchNorm1d(hidden_dim), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim), nn.BatchNorm1d(hidden_dim), nn.ReLU()
        )
    def forward(self, x): return self.net(x)

class FusionLayer(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Sequential(nn.Linear(hidden_dim * 2, 2), nn.Softmax(dim=-1))
        self.proj = nn.Linear(hidden_dim, hidden_dim)
    def forward(self, opt_emb, rad_emb):
        w = self.attn(torch.cat([opt_emb, rad_emb], dim=-1))
        return F.relu(self.proj(w[:, 0:1] * opt_emb + w[:, 1:2] * rad_emb))

class CMAGNN(nn.Module):
    def __init__(self, n_optical=5, n_radar=3, hidden_dim=64, n_classes=5, dropout=0.3, gat_heads=4):
        super().__init__()
        self.optical_branch = ModalityBranch(n_optical, hidden_dim, dropout)
        self.radar_branch   = ModalityBranch(n_radar, hidden_dim, dropout)
        self.fusion         = FusionLayer(hidden_dim)
        self.gat1 = GATConv(hidden_dim, hidden_dim // gat_heads, heads=gat_heads, dropout=dropout, concat=True)
        self.gat2 = GATConv(hidden_dim, hidden_dim, heads=1, dropout=dropout, concat=False)
        self.bn1  = nn.BatchNorm1d(hidden_dim)
        self.bn2  = nn.BatchNorm1d(hidden_dim)
        self.drop = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_dim, n_classes)

    def forward(self, x, edge_index):
        opt = self.optical_branch(x[:, :5])
        rad = self.radar_branch(x[:, 5:])
        h   = self.fusion(opt, rad)
        h   = self.drop(F.relu(self.bn1(self.gat1(h, edge_index))))
        h   = F.relu(self.bn2(self.gat2(h, edge_index)))
        return self.classifier(h)

model = CMAGNN().to(DEVICE)
model.load_state_dict(torch.load(os.path.join(OUTPUTS_DIR, 'best_model.pth'), map_location=DEVICE))
model.eval()
print('✅ Best model loaded.')

In [ ]:
# ── STEP 23: Test the Model ───────────────────────────────────
graph_data = torch.load(os.path.join(PROCESSED_DIR, 'graph_data.pt')).to(DEVICE)

with torch.no_grad():
    logits = model(graph_data.x, graph_data.edge_index)
    preds  = logits.argmax(dim=1)

y_true = graph_data.y[graph_data.test_mask].cpu().numpy()
y_pred = preds[graph_data.test_mask].cpu().numpy()

print(f'Test samples: {len(y_true):,}')

In [ ]:
# ── STEP 24: Confusion Matrix ────────────────────────────────

cm = confusion_matrix(y_true, y_pred)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title('Normalised Confusion Matrix — 2025 Landcover Classification')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUTS_DIR, 'confusion_matrix.png'), dpi=150)
plt.show()
print('✅ Confusion matrix saved.')

In [ ]:
# ── STEP 25-26: Accuracy Metrics ─────────────────────────────

oa  = accuracy_score(y_true, y_pred)
pa  = recall_score(y_true, y_pred, average=None, zero_division=0)      # Producer's (Recall)
ua  = precision_score(y_true, y_pred, average=None, zero_division=0)   # User's (Precision)
f1  = f1_score(y_true, y_pred, average=None, zero_division=0)

print('=' * 60)
print('CLASSIFICATION RESULTS — 2025')
print('=' * 60)
print(f'Overall Accuracy       : {oa * 100:.2f}%')
print(f'Avg Producer Accuracy  : {pa.mean() * 100:.2f}%')
print(f'Avg User Accuracy      : {ua.mean() * 100:.2f}%')
print(f'Avg F1-score           : {f1.mean() * 100:.2f}%')
print()

metrics_df = pd.DataFrame({
    'Class':              CLASS_NAMES,
    "Producer's Acc (%)": (pa * 100).round(2),
    "User's Acc (%)": (ua * 100).round(2),
    'F1-score (%)':       (f1 * 100).round(2)
})
print(metrics_df.to_string(index=False))

# Save
metrics_df.to_csv(os.path.join(OUTPUTS_DIR, 'classification_metrics_2025.csv'), index=False)
print('\n✅ Metrics saved to outputs/classification_metrics_2025.csv')